# Phase 3b — MMDetection Route B (native `mmdet` + RTMDet)

Rebuilds the detection stage on native **MMDetection** (RTMDet-s) instead of Ultralytics,
per the supervisor's ask, and integrates the enhancement/defogging transform as the
thesis's contribution.

Full background: `docs/MMDETECTION_ROUTE_B_RUNBOOK.md` (local-only, not pushed — see repo README).

**Run order:** mount Drive -> clone/pull repo -> §Env setup (Cells 1-7, run once per
session, does NOT survive a Colab disconnect) -> Task 2 (YOLO->COCO) -> Task 3 (config
sanity checks) -> Task 4 (train, costs compute) -> Task 6 (with/without eval, costs compute).

**Runtime:** Colab, **T4 GPU**. Set this before running anything: `Runtime > Change runtime type > T4 GPU`.


## 0. Mount Drive and get the repo

Datasets live on Drive, not in git. Code comes from git.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

REPO_DIR = '/content/computer_vision'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Ib-Programmer/computer_vision.git {REPO_DIR}

%cd {REPO_DIR}


Cloning into '/content/computer_vision'...
remote: Enumerating objects: 909, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 909 (delta 66), reused 57 (delta 31), pack-reused 792 (from 1)
Receiving objects: 100% (909/909), 47.56 MiB | 27.22 MiB/s, done.
Resolving deltas: 100% (606/606), done.
/content/computer_vision


In [3]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
ok = False
if nvidia_smi:
    try:
        r = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True)
        ok = r.returncode == 0 and bool(r.stdout.strip())
        if ok:
            print(r.stdout)
    except OSError:
        ok = False

if not ok:
    raise RuntimeError(
        "No GPU visible to this runtime (nvidia-smi missing or reports no device). Go to "
        "Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, then "
        "re-run from the top. (Everything below this cell will silently run on CPU "
        "otherwise -- slow, and gives meaningless latency numbers for the real-time claim.)"
    )


GPU 0: Tesla T4 (UUID: GPU-95bc63fa-2154-d97d-4be9-65b60f3e9627)



### Recovery cell — run this any time paths/state look wrong

Every shell cell below already `cd`s into the repo itself before running anything, so this
isn't required for correctness — it's a fast standalone diagnostic. Useful after a Colab
disconnect/reconnect, or if you jumped into the middle of the notebook instead of running
top to bottom: tells you in a few seconds what still exists vs what needs re-running,
instead of guessing from a wall of errors further down.


In [ ]:
import os
import subprocess

REPO_DIR = '/content/computer_vision'
MMDET_REPO = '/content/mmdetection'

get_ipython().run_line_magic('cd', REPO_DIR) if os.path.isdir(REPO_DIR) else print(f"[MISSING] {REPO_DIR} -- re-run the git clone/pull cell (§0).")

checks = [
    ("repo checked out", os.path.isdir(REPO_DIR)),
    ("mmdetection tools/ cloned (v3.3.0)", os.path.isdir(f"{MMDET_REPO}/tools")),
    ("mm conda env exists", os.path.isdir("/usr/local/envs/mm")),
    ("COCO annotations converted (Task 2)", os.path.exists(f"{REPO_DIR}/datasets/bdd100k_yolo/annotations/train.json")),
    ("Task 4 checkpoint exists", os.path.exists(f"{REPO_DIR}/work_dirs/rtmdet_bdd100k/latest.pth")),
]
for label, present in checks:
    print(f"[{'OK' if present else 'MISSING'}] {label}")

if os.path.isdir("/usr/local/envs/mm"):
    r = subprocess.run(['conda', 'run', '-n', 'mm', 'python', '-c',
                         "import torch; print('CUDA:', torch.cuda.is_available())"],
                        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())


## 1. Environment setup (§3 of the runbook) — run once per session

Colab's default runtime (Python 3.12, torch 2.11, CUDA 12.8) is **incompatible** with the
OpenMMLab 2.x stack. `condacolab` gives us conda, then we build a separate **Python 3.10**
conda env (`mm`) with a pinned stack. The kernel itself stays 3.12 — every mmdet call below
routes through `conda run -n mm`.

Do not deviate from the pinned versions (torch 2.1.0+cu118 / mmcv 2.1.0 / mmdet 3.3.0 /
numpy<2) — see runbook §3 for why each pin exists.


In [4]:
# CELL 1 — run ALONE. The kernel auto-restarts after this (expected). Do not re-run.
!pip install -q condacolab
import condacolab
condacolab.install()


⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...


In [ ]:
# Cell 1's condacolab.install() restarted the kernel, which resets the working directory
# back to /content -- the %cd into the repo from the clone/pull cell above does NOT
# survive that restart. Re-cd here so every relative path in the rest of this notebook
# (scripts/..., configs/..., datasets/...) resolves correctly.
%cd /content/computer_vision


In [5]:
# CELL 2 — after the restart: create the 3.10 env
!mamba create -n mm python=3.10 -y || conda create -n mm python=3.10 -y


[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   6%
conda-forge/noarch    16%[+] 0.3s
conda-forge/linux-64  14%
conda-forge/noarch    31%[+] 0.4s
conda-forge/linux-64  20%
conda-forge/noarch    43%[+] 0.5s
conda-forge/linux-64  26%
conda-forge/noarch    53%[+] 0.6s
conda-forge/linux-64  31%
conda-forge/noarch    64%[+] 0.7s
conda-forge/linux-64  33%
conda-forge/noarch    74%[+] 0.8s
conda-forge/linux-64  39%
conda-forge/noarch    85%[+] 0.9s
conda-forge/linux-64  43%
conda-forge/noarch    94%conda-forge/noarch                                
[+] 1.0s
conda-forge/linux-64  49%[+] 1.1s
conda-forge/linux-64  57%[+] 1.2s
conda-forge/linux-64  63%[+] 1.3s
conda-forge/linux-64  66%[+] 1.4s
conda-forge/linux-64  72%[+] 1.5s
conda-forge/linux-64  78%[+] 1.6s
conda-forge/linux-64  78%[+] 1.7s
conda-forge/linux-64  78%[+] 1.8s
conda-forge/linux-64  79%[+] 1.9s
conda-forge/linux-64  79%[+] 2.0s
conda-forge/linux-64  79%[+] 2.1s
conda-forge/linux

In [6]:
# CELL 3 — GATE: must print 3.10.x before continuing. Stop here if it doesn't.
!conda run -n mm python -c "import sys; print('env Python:', sys.version.split()[0])"


env Python: 3.10.20



In [7]:
# CELL 4 — pinned torch (cu118 runs fine under the T4's 12.8 driver)
!conda run -n mm pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118
!conda run -n mm pip install -q "numpy<2"
!conda run -n mm python -c "import numpy,torch; print('numpy',numpy.__version__,'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())"


numpy 1.26.4 | torch 2.1.0+cu118 | CUDA True



In [8]:
# CELL 5 — OpenMMLab stack (mmcv from the matching prebuilt index)
!conda run -n mm pip install -q -U openmim
!conda run -n mm mim install mmengine
!conda run -n mm mim install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!conda run -n mm mim install "mmdet==3.3.0"


Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 73.9 MB/s  0:00:00
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 136.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 146.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.7 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html, https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 MB 5.0 MB/s  0:00:19


A module that was compiled us

In [9]:
# CELL 6 — RE-PIN numpy: installing the stack drags numpy back to 2.x, which breaks it.
!conda run -n mm pip install -q "numpy<2"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.



In [1]:
# CELL 7 — smoke test. Success = "OK -- native MMDetection works."
!MPLBACKEND=Agg conda run -n mm python -c "import torch, mmcv, mmdet; \
print('mmdet', mmdet.__version__, '| CUDA', torch.cuda.is_available()); \
from mmdet.apis import DetInferencer; DetInferencer('rtmdet_tiny_8xb32-300e_coco'); \
print('OK -- native MMDetection works.')"


mmdet 3.3.0 | CUDA True
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

08/13 15:25:17 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.
OK -- native MMDetection works.

Downloading: "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth" to /root/.cache/torch/hub/checkpoints/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
/usr/local

In [ ]:
# `pip install mmdet` does NOT ship tools/train.py or tools/test.py as an importable
# submodule -- `python -m mmdet.tools.train` can never work (confirmed:
# ModuleNotFoundError: No module named 'mmdet.tools'). Those scripts only exist in the
# mmdetection source repo. Clone the tag matching our pinned mmdet==3.3.0 once and call
# the scripts by absolute path instead (used by Task 3's overfit check and Tasks 4/6 below).
import os

MMDET_REPO = '/content/mmdetection'
if not os.path.isdir(MMDET_REPO):
    !git clone --depth 1 --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git {MMDET_REPO}


### Optional — snapshot the env so you don't rebuild it every session

The `mm` env does **not** survive a Colab disconnect. Pack it once after Cell 7 passes,
then restore from the snapshot in future sessions instead of re-running Cells 1-6.


In [ ]:
# Save (run once, after Cell 7 passes). Takes a while; ~2-4 GB on Drive.
# conda-pack must be installed in the OUTER/base env (it invokes `conda pack`, not
# `python -m conda_pack`) -- installing it into `mm` via `conda run -n mm pip install`
# (the original bug here) leaves the base `conda` CLI without the `pack` subcommand.
# --ignore-missing-files: the pip installs throughout setup overwrote some files conda
# itself originally laid down (e.g. packaging, setuptools) -- conda-pack refuses to pack
# by default when it detects that; we don't need byte-for-byte provenance for a dev snapshot.
!mkdir -p /content/drive/MyDrive/computer_vision
!pip install -q conda-pack
!conda pack -n mm -o /content/drive/MyDrive/computer_vision/mm_env.tar.gz --ignore-missing-files


In [5]:
import os

SNAPSHOT = '/content/drive/MyDrive/computer_vision/mm_env.tar.gz'
if not os.path.exists(SNAPSHOT):
    print(f"[WARN] no snapshot at {SNAPSHOT} yet -- run the save cell above first (once, "
          f"after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.")
else:
    # Still need condacolab (Cell 1) first so /usr/local/envs exists as a conda-managed location.
    get_ipython().system('mkdir -p /usr/local/envs/mm')
    get_ipython().system(f'tar -xzf {SNAPSHOT} -C /usr/local/envs/mm')
    get_ipython().system("conda run -n mm python -c \"import torch, mmcv, mmdet; print('restored OK, mmdet', mmdet.__version__)\"")


[WARN] no snapshot at /content/drive/MyDrive/computer_vision/mm_env.tar.gz yet -- run the save cell above first (once, after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.


## 2. Task 2 — YOLO -> COCO conversion

Reads `datasets/bdd100k_yolo/{train,val}/images|labels` (confirmed on-Drive layout, see
`scripts/preprocess_data.py`) and writes `datasets/bdd100k_yolo/annotations/{train,val}.json`.

Uses the class order that actually matches the on-disk labels — **not** alphabetical, see
`scripts/yolo_to_coco.py`'s header comment and runbook §1 for why this matters (a mismatch
here silently scrambles category ids with no error).


In [ ]:
!cd /content/computer_vision && conda run -n mm python scripts/yolo_to_coco.py


In [ ]:
%%bash
cd /content/computer_vision
# Acceptance check: pycocotools loads both files without error, and annotation count
# roughly matches non-empty label-file line count.
conda run -n mm python << 'PY'
from pycocotools.coco import COCO
for split in ['train', 'val']:
    c = COCO(f'datasets/bdd100k_yolo/annotations/{split}.json')
    print(split, '-> images:', len(c.imgs), '| annotations:', len(c.anns), '| categories:', len(c.cats))
PY


## 3. Task 3 — RTMDet config sanity checks

Before trusting `configs/rtmdet_bdd100k.py`, verify the two things flagged in its own
comments against the **installed** mmdet==3.3.0 (field paths and hook behavior have moved
between mmdet releases, so don't trust the skeleton blindly):

1. `bbox_head.num_classes` field path resolves correctly on the base RTMDet-s config.
2. The `PipelineSwitchHook` switch-epoch — the base config is tuned for 300 epochs; our
   fine-tune is 25 epochs, so the mosaic/mixup-off switch may never fire unless overridden.


In [8]:
%%bash
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py')
print('base bbox_head.num_classes:', c.model.bbox_head.num_classes)
for hook in c.custom_hooks:
    if 'PipelineSwitch' in hook.get('type', ''):
        print('PipelineSwitchHook switch_epoch:', hook.get('switch_epoch'))
PY


In [ ]:
%%bash
cd /content/computer_vision
# Loads our actual fine-tune config and confirms num_classes took effect (should be 10).
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('fine-tune bbox_head.num_classes:', c.model.bbox_head.num_classes)
print('max_epochs:', c.train_cfg.max_epochs)
PY


### 1-image overfit sanity check (cheap, ~1-2 min on T4)

Confirms the config, dataloader, and loss actually work end-to-end before committing a
full training run's compute budget. Loss should visibly decrease over a handful of iters.


In [ ]:
# TODO before running: point train_dataloader at a 1-image subset, e.g. by adding
#   train_dataloader = dict(dataset=dict(indices=1))
# to a throwaway copy of the config, or pass --cfg-options train_dataloader.dataset.indices=1
# on the command line if your mmdet build's train.py accepts --cfg-options (mmdet 3.x does).
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py \
    --cfg-options train_dataloader.dataset.indices=1 train_cfg.max_epochs=1 train_cfg.val_interval=1


## 4. Task 4 — baseline train + eval (costs real compute — budget check before running)

Reproduces the Phase 3 baseline inside mmdet. Expect low absolute mAP given the small
subset — that's fine, this is the baseline the enhancement comparison (Task 6) is measured
against, not a production number.


In [ ]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py


## 5. Task 6 — with/without enhancement evaluation (the thesis result)

Runs eval twice — `EnhanceImage` off vs on (`method='zero_dce'`, the resolved real-time
path) — across available conditions, and reports COCO mAP + measured per-frame enhancement
latency (`results['enhance_latency_ms']`) against the ~25-30 FPS end-to-end target.

Wire `EnhanceImage` into `test_pipeline` (see `scripts/mm_transforms.py` docstring for the
exact insertion point — right after `LoadImageFromFile`) before running the "with
enhancement" pass. Keep a copy of the config without it for the "without" baseline pass.


In [ ]:
# Baseline (no enhancement) — uses configs/rtmdet_bdd100k.py + the checkpoint from Task 4.
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_baseline.pkl


In [ ]:
# With enhancement — point at a config variant that adds EnhanceImage to test_pipeline
# (e.g. configs/rtmdet_bdd100k_enhanced.py, once you've created it as a small delta config
# with custom_imports=['scripts.mm_transforms'] and the EnhanceImage insertion).
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k_enhanced.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl
